# 👥 Customers Dataset — Cleaning & Feature Engineering
---
**Dataset:** `customers.csv`  
**Goal:** Clean data, validate quality, engineer useful features for analysis.  
**Final output:** `customers_cleaned.csv`

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

## 2. Load Dataset

In [2]:
df = pd.read_csv("customers.csv")

print("Shape:", df.shape)
df.head()

Shape: (5000, 11)


,customer_id,customer_name,age,gender,city,signup_date,customer_segment,total_orders,avg_order_value,preferred_payment,is_active
0,1,Keya Bora,52,Male,Lucknow,2024-04-04,Regular,6,464,Wallet,Yes
1,2,Misha Manda,53,Male,Kolkata,2025-09-09,Inactive,1,156,Wallet,No
2,3,Dhanuk Ganesh,45,Female,Pune,2024-02-01,Premium,63,1477,Credit Card,Yes
3,4,Pihu Bera,56,Female,Mumbai,2024-08-13,Regular,16,399,Wallet,Yes
4,5,Pihu Aggarwal,54,Male,Delhi,2024-01-10,New,5,341,UPI,Yes


## 3. Explore the Data

In [6]:
df.describe()

,customer_id,age,total_orders,avg_order_value
count,5000.000000,5000.000000,5000.000000,5000.000000
mean,2500.500000,38.943800,18.336800,560.187800
std,1443.520003,12.370742,19.941798,318.778499
min,1.000000,18.000000,0.000000,150.000000
25%,1250.750000,28.000000,4.000000,349.750000
50%,2500.500000,39.000000,12.000000,464.000000
75%,3750.250000,49.000000,23.000000,659.000000
max,5000.000000,60.000000,80.000000,1499.000000


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   customer_id        5000 non-null   int64
 1   customer_name      5000 non-null   str  
 2   age                5000 non-null   int64
 3   gender             5000 non-null   str  
 4   city               5000 non-null   str  
 5   signup_date        5000 non-null   str  
 6   customer_segment   5000 non-null   str  
 7   total_orders       5000 non-null   int64
 8   avg_order_value    5000 non-null   int64
 9   preferred_payment  5000 non-null   str  
 10  is_active          5000 non-null   str  
dtypes: int64(4), str(7)
memory usage: 429.8 KB


## 4. Check Duplicates & Missing Values

In [3]:
# Check for fully duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check null values column-wise
print("\nNull values per column:")
print(df.isnull().sum())

Duplicate rows: 0

Null values per column:
customer_id          0
customer_name        0
age                  0
gender               0
city                 0
signup_date          0
customer_segment     0
total_orders         0
avg_order_value      0
preferred_payment    0
is_active            0
dtype: int64


## 5. Data Validation

In [4]:
# Age should realistically be between 18 and 100 for a food delivery app
invalid_age = df[(df['age'] < 18) | (df['age'] > 100)]
print(f"Invalid age rows: {len(invalid_age)}")

Invalid age rows: 0


In [5]:
# Convert signup_date to datetime first
df['signup_date'] = pd.to_datetime(df['signup_date'])

In [6]:
# Signup date cannot be in the future
future_signups = df[df['signup_date'] > pd.Timestamp.today()]
print(f"Future signup dates (invalid): {len(future_signups)}")

Future signup dates (invalid): 0


In [7]:
# Remove rows with future signup dates — they are data entry errors
df = df[df['signup_date'] <= pd.Timestamp.today()].copy()
print(f"Rows remaining after removing invalid signup dates: {len(df)}")

Rows remaining after removing invalid signup dates: 5000


## 6. Feature Engineering

In [8]:
# Bin customers into age brackets for segment-level analysis
df['age_group'] = pd.cut(
    df['age'],
    bins=[18, 25, 35, 45, 60],
    labels=['18-25', '26-35', '36-45', '46-60'],
    right=True
)

In [9]:
# Customers with avg_order_value > 800 are considered high value
df['high_value_customer'] = np.where(
    df['avg_order_value'] > 800,
    'Yes', 'No'
)

In [10]:
# How long has the customer been with the platform
df['customer_tenure_days'] = (
    pd.Timestamp.today() - df['signup_date']
).dt.days

## 7. Final Check & Export

In [11]:
# Final shape and null check before saving
print("Final Shape:", df.shape)
print("\nNull values after all cleaning steps:")
print(df.isnull().sum())
df.head()

Final Shape: (5000, 14)

Null values after all cleaning steps:
customer_id               0
customer_name             0
age                       0
gender                    0
city                      0
signup_date               0
customer_segment          0
total_orders              0
avg_order_value           0
preferred_payment         0
is_active                 0
age_group               108
high_value_customer       0
customer_tenure_days      0
dtype: int64


,customer_id,customer_name,age,gender,city,signup_date,customer_segment,total_orders,avg_order_value,preferred_payment,is_active,age_group,high_value_customer,customer_tenure_days
0,1,Keya Bora,52,Male,Lucknow,2024-04-04,Regular,6,464,Wallet,Yes,46-60,No,766
1,2,Misha Manda,53,Male,Kolkata,2025-09-09,Inactive,1,156,Wallet,No,46-60,No,243
2,3,Dhanuk Ganesh,45,Female,Pune,2024-02-01,Premium,63,1477,Credit Card,Yes,36-45,Yes,829
3,4,Pihu Bera,56,Female,Mumbai,2024-08-13,Regular,16,399,Wallet,Yes,46-60,No,635
4,5,Pihu Aggarwal,54,Male,Delhi,2024-01-10,New,5,341,UPI,Yes,46-60,No,851


In [12]:
df.to_csv("customers_cleaned.csv", index=False)
print("✅ customers_cleaned.csv saved successfully!")

✅ customers_cleaned.csv saved successfully!
